In [25]:
%pip install pandas scipy
from pathlib import Path
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.signal

audio_path = Path("../data/data_raw/learner01_52.wav")
y, sr = librosa.load(audio_path, sr=None)

print("Sample rate :", sr)
print("Jumlah sample:", len(y))
print("Durasi       :", len(y) / sr, "detik")


[notice] A new release of pip is available: 23.3.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Sample rate : 32000
Jumlah sample: 268800
Durasi       : 8.4 detik


In [26]:

def extract_mfcc_features(path, sr=16000, n_mfcc=13):
    """Load file, ekstrak MFCC, Delta, dan Delta-Delta, lalu agregasi."""
    # 1. Load + resample
    y, _ = librosa.load(path, sr=sr)

    # 2. Trim silence
    y, _ = librosa.effects.trim(y, top_db=30)

    # 3. MFCC Dasar -> shape (13, n_frames)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    
    # 4. Delta (Turunan pertama) dan Delta-Delta (Turunan kedua)
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
    
    # 5. Gabungkan menjadi 39 koefisien (13 asli + 13 delta + 13 delta-delta)
    mfcc_all = np.vstack([mfcc, mfcc_delta, mfcc_delta2])

    # 6. Agregasi mean dan std (TANPA CMVN)
    features = np.concatenate([mfcc_all.mean(axis=1), mfcc_all.std(axis=1)])

    # 7. Penamaan kolom
    cols = []
    for f_type in ['mfcc', 'delta', 'delta2']:
        cols.extend([f'{f_type}{i+1}_mean' for i in range(n_mfcc)])
    for f_type in ['mfcc', 'delta', 'delta2']:
        cols.extend([f'{f_type}{i+1}_std' for i in range(n_mfcc)])
        
    return dict(zip(cols, features))



In [27]:
import pandas as pd
import numpy as np
from pathlib import Path

AUDIO_DIR = Path('../data/data_raw')          # <-- sesuaikan folder audio mentah kamu
EXT = '.wav'                        # <-- sesuaikan kalau .m4a/.mp3

meta_speaker = pd.read_csv('../data/metadata.csv')
CODES = ['52', '63', '64', '68', '95']

rows = []
for _, r in meta_speaker.iterrows():
    for code in CODES:
        fpath = AUDIO_DIR / f"{r['speaker_id']}_{code}{EXT}"
        fpath_processed = Path('../data/data_preprocessed') / f"{r['speaker_id']}_{code}{EXT}"
        if not fpath.exists():
            print(f"⚠️  tidak ketemu: {fpath}")
            continue
        rows.append({
            'file_path': str(fpath),
            'file_path_processed': str(fpath_processed),
            'speaker_id': r['speaker_id'],
            'class': r['class'],
            'gender': r['gender'],
            'location': r['recording_location'],
            'notes': r['notes'],
        })

meta = pd.DataFrame(rows)
print(meta.shape)   
assert meta['class'].value_counts().to_dict() == {'learner': 30, 'native': 30}
meta.to_csv('metadata_utterance.csv', index=False)

(60, 7)


In [28]:
import librosa
import soundfile as sf

OUT_DIR = Path('../data/data_preprocessed'); OUT_DIR.mkdir(exist_ok=True)
SR = 16000

def preprocess(y, sr):
    # 1. noise reduction ringan khusus file kafe (opsional, pakai noisereduce)
    
    try:
        import noisereduce as nr
        y = nr.reduce_noise(y=y, sr=sr, stationary=True, prop_decrease=0.8)
    except ImportError:
            # fallback: high-pass filter buang low-frequency rumble kafe
        y = librosa.effects.preemphasis(y)
    # 2. trim silence (top_db lebih tinggi buat file kafe supaya
    #    noise ambient gak kebaca sebagai speech)
    top_db = 40 
    y, _ = librosa.effects.trim(y, top_db=top_db)
    # 3. resample ke 16kHz
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    # 4. RMS normalization → target RMS 0.1
    rms = np.sqrt(np.mean(y**2)) + 1e-8
    y = y * (0.1 / rms)
    return y

for _, r in meta.iterrows():
    y, sr = librosa.load(r['file_path'], sr=None)
    y = preprocess(y, sr)
    sf.write(OUT_DIR / Path(r['file_path']).name, y, SR)

In [29]:
%pip install praat-parselmouth
import parselmouth
from parselmouth.praat import call

def extract_prosody(path, sr=16000):
    snd = parselmouth.Sound(path)
    feats = {}

    # --- Durasi Aktif & Speaking Rate Proxy ---
    y, _ = librosa.load(path, sr=sr)
    y_trimmed, _ = librosa.effects.trim(y, top_db=30)
    # Rasio panjang suara asli setelah dibuang silence-nya
    feats['voiced_duration'] = len(y_trimmed) / sr
    # Menghitung perkiraan jumlah suku kata/mora (onset detection)
    onset_env = librosa.onset.onset_strength(y=y_trimmed, sr=sr)
    peaks, _ = scipy.signal.find_peaks(onset_env, height=np.mean(onset_env))
    
    # Rasio jumlah ketukan per detik bersuara
    feats['syllable_rate'] = len(peaks) / feats['voiced_duration'] if feats['voiced_duration'] > 0 else 0

    # --- F0 (pitch) ---
    pitch = call(snd, "To Pitch", 0.0, 75, 600)
    f0 = pitch.selected_array['frequency']
    f0 = f0[f0 > 0]  # buang unvoiced frame
    
    feats['f0_mean'] = f0.mean() if len(f0) else np.nan
    feats['f0_std']  = f0.std() if len(f0) else np.nan
    feats['f0_range'] = (f0.max() - f0.min()) if len(f0) else np.nan
    
    # TAMBAHAN: Pergerakan Pitch / Intonasi (Delta F0)
    if len(f0) > 1:
        f0_delta = np.diff(f0)
        feats['f0_delta_mean_abs'] = np.abs(f0_delta).mean() # Rata-rata lompatan nada absolut
        feats['f0_delta_std'] = f0_delta.std()
    else:
        feats['f0_delta_mean_abs'] = np.nan
        feats['f0_delta_std'] = np.nan

    # --- Jitter & Shimmer ---
    point = call(snd, "To PointProcess (periodic, cc)", 75, 600)
    feats['jitter_local'] = call(point, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    feats['shimmer_local'] = call([point, snd], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.2)

    # --- Formant F1, F2 ---
    formant = call(snd, "To Formant (burg)", 0.0, 5, 5500, 0.025, 50)
    f1_vals, f2_vals = [], []
    for t in np.arange(0.1, snd.duration - 0.1, 0.05):
        f1 = call(formant, "Get value at time", 1, t, 'Hertz', 'Linear')
        f2 = call(formant, "Get value at time", 2, t, 'Hertz', 'Linear')
        if not np.isnan(f1): f1_vals.append(f1)
        if not np.isnan(f2): f2_vals.append(f2)
    feats['f1_mean'] = np.mean(f1_vals) if f1_vals else np.nan
    feats['f2_mean'] = np.mean(f2_vals) if f2_vals else np.nan
    feats['f1f2_ratio'] = feats['f1_mean'] / feats['f2_mean'] if f1_vals and f2_vals else np.nan

    return feats


[notice] A new release of pip is available: 23.3.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
%pip install tqdm
from tqdm import tqdm

def extract_all(path):
    feats = extract_mfcc_features(path)   # fungsi langkah 5 sebelumnya
    feats.update(extract_prosody(path))  # fungsi langkah 6 sebelumnya
    return feats

records = []
for _, r in tqdm(meta.iterrows(), total=len(meta)):
    feats = extract_all(r['file_path_processed'])
    feats.update(r[['speaker_id', 'class', 'gender', 'location']].to_dict())
    records.append(feats)

features = pd.DataFrame(records)
features.to_csv('features.csv', index=False)
print(features.shape)  # (55, ~35 fitur)


[notice] A new release of pip is available: 23.3.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


100%|██████████| 60/60 [00:17<00:00,  3.35it/s]

(60, 94)


In [31]:
prosody_rows = [extract_prosody(r['file_path_processed']) for _, r in meta.iterrows()]
df_prosody = pd.DataFrame(prosody_rows)
df_prosody['speaker_id'] = meta['speaker_id'].values
df_prosody['class'] = meta['class'].values

mfcc_rows = [
    extract_mfcc_features(r['file_path_processed'])
    for _, r in meta.iterrows()
]

df_mfcc = pd.DataFrame(mfcc_rows)
df_mfcc['speaker_id'] = meta['speaker_id'].values
df_mfcc['class'] = meta['class'].values

# Gabungkan berdasarkan baris, bukan hanya speaker_id/class,
# karena setiap speaker memiliki beberapa file audio.
features = pd.concat(
    [
        df_mfcc.reset_index(drop=True),
        df_prosody.drop(columns=['speaker_id', 'class']).reset_index(drop=True)
    ],
    axis=1
)

features.to_csv('features.csv', index=False)
print(features.shape)  # (55, ~35 fitur + 2 kolom id)

(60, 92)
